# Lunar Lander — DQN / Double DQN

Entraînement d'un agent DQN sur LunarLander. Switch `DOUBLE_DQN = True` pour passer en Double DQN.

In [ ]:
import os, time
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import trange

from src.agent import DQNAgent
from src.utils import set_seed, make_env, record_episode, evaluate, LUNAR_ENV_ID

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
SEED = 42
MAX_EPISODES = 1000
MAX_STEPS_PER_EPISODE = 1000
DOUBLE_DQN = False
EPS_SCHEDULE = 'linear'
TARGET_UPDATE_FREQ = 500
RECORD_EVERY = 200

NAME = 'double_dqn' if DOUBLE_DQN else 'dqn'
os.makedirs('outputs/checkpoints', exist_ok=True)
os.makedirs('outputs/videos', exist_ok=True)
os.makedirs('outputs/plots', exist_ok=True)

set_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
env = make_env(seed=SEED)
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print('state_dim:', state_dim, '| n_actions:', n_actions)

agent = DQNAgent(
    state_dim=state_dim,
    n_actions=n_actions,
    eps_schedule=EPS_SCHEDULE,
    target_update_freq=TARGET_UPDATE_FREQ,
    double_dqn=DOUBLE_DQN,
    device=device,
)

### Vidéo de l'agent au début (pas encore entraîné)

In [ ]:
r0 = record_episode(agent, f'outputs/videos/{NAME}_start.gif', seed=SEED)
print(f'random agent reward: {r0:.1f}')

### Training loop

In [ ]:
rewards = []
losses = []
window = deque(maxlen=100)
best_avg = -np.inf
t0 = time.time()

pbar = trange(MAX_EPISODES)
for ep in pbar:
    s, _ = env.reset(seed=SEED + ep)
    total_reward = 0.0
    ep_losses = []
    for _ in range(MAX_STEPS_PER_EPISODE):
        a = agent.act(s)
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        agent.push(s, a, r, s_next, float(terminated))
        loss = agent.train_step()
        if loss is not None:
            ep_losses.append(loss)
        agent.increment_step()
        s = s_next
        total_reward += r
        if done:
            break
    rewards.append(total_reward)
    window.append(total_reward)
    avg = np.mean(window)
    if ep_losses:
        losses.append(np.mean(ep_losses))
    pbar.set_description(f'ep {ep} | r {total_reward:7.1f} | avg100 {avg:7.1f} | eps {agent.eps:.3f}')
    if len(window) >= 50 and avg > best_avg:
        best_avg = avg
        agent.save(f'outputs/checkpoints/{NAME}_best.pt')
    if RECORD_EVERY and ep > 0 and ep % RECORD_EVERY == 0:
        record_episode(agent, f'outputs/videos/{NAME}_ep{ep}.gif', seed=SEED + 9999)
    if avg >= 200 and len(window) == 100:
        print(f'\nresolved at episode {ep} (avg100 = {avg:.1f})')
        break

agent.save(f'outputs/checkpoints/{NAME}_final.pt')
np.save(f'outputs/checkpoints/{NAME}_rewards.npy', np.array(rewards))
print(f'training time: {(time.time() - t0)/60:.1f} min')

### Courbes d'apprentissage

In [ ]:
def moving_avg(x, w=100):
    x = np.array(x)
    if len(x) < w:
        return x
    c = np.cumsum(np.insert(x, 0, 0))
    return (c[w:] - c[:-w]) / w

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(rewards, alpha=0.3, label='episode reward')
ma = moving_avg(rewards, 100)
ax[0].plot(np.arange(len(ma)) + 100, ma, label='moving avg (100)')
ax[0].axhline(200, color='r', linestyle='--', alpha=0.5, label='solved')
ax[0].set_xlabel('Episode'); ax[0].set_ylabel('Reward')
ax[0].legend(); ax[0].set_title(f'{NAME} — reward')

ax[1].plot(losses)
ax[1].set_xlabel('Episode'); ax[1].set_ylabel('Loss (mean per ep)')
ax[1].set_yscale('log'); ax[1].set_title('Loss')

plt.tight_layout()
plt.savefig(f'outputs/plots/{NAME}_training.png', dpi=120)
plt.show()

### Évaluation et vidéos finales

In [ ]:
mean_r, std_r, all_r = evaluate(agent, n_episodes=20, seed=999)
print(f'eval over 20 ep: {mean_r:.1f} ± {std_r:.1f}')
print('min/max:', min(all_r), '/', max(all_r))

In [ ]:
record_episode(agent, f'outputs/videos/{NAME}_final.gif', seed=42)
record_episode(agent, f'outputs/videos/{NAME}_final.mp4', seed=42)
print('vidéos finales enregistrées')

### Comparaison DQN vs Double DQN

Si tu as déjà entraîné les deux (ie. lancé ce notebook avec `DOUBLE_DQN = False` puis `True`), tu peux superposer les courbes.

In [ ]:
import os.path as op
fig, ax = plt.subplots(figsize=(9, 4))
for tag in ['dqn', 'double_dqn']:
    path = f'outputs/checkpoints/{tag}_rewards.npy'
    if op.exists(path):
        r = np.load(path)
        ma = moving_avg(r, 100)
        ax.plot(np.arange(len(ma)) + 100, ma, label=tag.upper())
ax.axhline(200, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Reward (MA100)')
ax.legend(); ax.set_title('DQN vs Double DQN')
plt.tight_layout()
plt.savefig('outputs/plots/dqn_vs_double.png', dpi=120)
plt.show()